# Zarr Stores with Xarray

## Learning Objectives:
- Learn about the Zarr data format and xarray's "zarr" backend engine
- Learn how to read a Zarr store with a hierarchical structure with `xr.DataTree`
- Learn how to select dask arrays from a Zarr store
- Explore how to use Zarr stores with xarray for computations and visualizations

## What is Zarr?

The Zarr data format is an open, community-maintained format designed for efficient, scalable storage of large N-dimensional arrays. It stores data as compressed and chunked arrays in a format well-suited to parallel processing and cloud-native workflows. Xarray’s Zarr backend allows xarray to leverage these capabilities, including the ability to store and analyze datasets far too large fit onto disk (particularly in combination with [dask](https://docs.xarray.dev/en/latest/user-guide/dask.html#dask)).

### Zarr Data Organization:
- **Arrays**: N-dimensional arrays that can be chunked and compressed.
- **Groups**: A container for organizing multiple arrays and other groups with a hierarchical structure.
- **Metadata**: JSON-like metadata describing the arrays and groups, including information about data types, dimensions, chunking, compression, and user-defined key-value fields. 
- **Dimensions and Shape**: Arrays can have any number of dimensions, and their shape is defined by the number of elements in each dimension.
- **Coordinates & Indexing**: Zarr supports coordinate arrays for each dimension, allowing for efficient indexing and slicing.

The diagram below from [the Zarr v3 specification](https://wiki.earthdata.nasa.gov/display/ESO/Zarr+Format) showing the structure of a Zarr store:

![ZarrSpec](https://zarr-specs.readthedocs.io/en/latest/_images/terminology-hierarchy.excalidraw.png)


NetCDF and Zarr share similar terminology and functionality, but the key difference is that NetCDF is a single file, while Zarr is a directory-based “store” composed of many chunked files, making it better suited for distributed and cloud-based workflows.

## Reading a zarr store
With xarray's "zarr" backend, we can read data remotely from cloud storage buckets. Since Zarr is a cloud-optimized format, our dataset is lazily-loaded as dask arrays.
The Zarr store in this tutorial is in a public storage bucket and is a precipitation dataset with a group hierarchical structure. The groups: "observed" and "reanalysis", each contain a "precipitation" data variable derived from *[GPM_3IMERGHH_07](https://disc.gsfc.nasa.gov/datasets/GPM_3IMERGHH_07/summary)* and *[M2T1NXFLX_5.12.4](https://disc.gsfc.nasa.gov/datasets/M2T1NXFLX_5.12.4/summary)* products, respectively. The precipitation data variables in each group have different spatial resolutions that cannot be represented in a single group, so we need to read them as `DataTree` objects.

Let's read in our zarr store with `xr.open_datatree`

In [ ]:
import xarray as xr

precipitation_store = "https://pub-45a1d62ac8d94c4c89f4dc63681a98ed.r2.dev/precipitation.zarr"

precip_dt = xr.open_datatree(precipitation_store, engine="zarr", chunks={}, consolidated=True)

In [ ]:
precip_dt.observed.precipitation

:::{note} We selected `"zarr"` backend engine, which tells xarray to load and decode a dataset from a Zarr store. The `chunks={}` parameter is used to load the data into a dask array. And `consolidated=True` enables zarr’s consolidated metadata capability. This lets us read all of the metadata from a single file which can improve performance.
:::

## Variable selection
With our `DataTree` object, we can select variables from our Zarr store with either dictionary and or attribute like syntax.

### Dictionary-like interface

In [ ]:
precip_dt["observed/precipitation"]

### Attribute-like access

In [ ]:
precip_dt.observed.precipitation

### Dictionary and Attribute like access

In [ ]:
precip_dt.observed["precipitation"]

:::{note} All of these variable selection options return the same "precipitation" `xr.DataArray` object, as a chunked `dask.Array`, from the "observed" group of our Zarr store.
:::

## Time slicing

We can index and subset by time on our `xr.DataTree` object. Each time slice in our Zarr store represents one hour of data with a total of 10 hours of data.

Let's explore the different ways we can get the first 5 hours of data. 

### Label-based indexing

Let's try getting the first 5 hours of data with `.sel(time=)`. 

Since the time slices are ordered we can get a subset of the array of our time coordinate and pass it to the `.sel` method.

In [ ]:
time_index = precip_dt.time[0:5]
precip_dt.sel(time=time_index)

### Datetime indexing
We can also subset by time with a `datetime` string.

In [ ]:
precip_dt.sel(time=slice("2021-08-29T07:30:00", "2021-08-29T16:30:00"))

### Positional Indexing
Or by the index of our time dimension `.isel(time=slice())`

In [ ]:
precip_dt.isel(time=slice(0, 5))

## Chunking
Chunking is the process of dividing arrays into smaller pieces, which allows for parallel processing and efficient storage.

To examine the chunks in our Zarr store, with `xarray` you can use the `chunks` attribute on the `xr.DataArray` object.

In [ ]:
precip_dt.observed["precipitation"].data.chunks

### Selecting by chunks

Since we loaded our data as a `dask.Array`, we can access data from each chunked array in our Zarr store. 

Let's get the first chunk of the "observed/precipitation" variable in our zarr store.

In [ ]:
precip_dt.observed["precipitation"].data.blocks[0, 0, 0].compute()

:::{note}
We added `.data` to our `xr.DataArray` to access the `dask.Array`. The `.blocks[]` method allows you to index by chunk and `.compute()` returns the `np.ndarray`. 
:::

## Exercise

::::{admonition} Exercise
:class: tip

Can you calculate and plot the mean precipitation, starting at 09:55 for the reanalysis group in this zarr store?

:::{admonition} Hint
:class: dropdown
This is how you could calculate mean "precipitation"

```python
precip_dt.reanalysis['precipitation'].mean(dim='time')
```
:::

:::{admonition} Solution
:class: dropdown

```python
precip_dt.reanalysis['precipitation'].sel(time=slice('2021-08-29T09:55:00', '2021-08-29T16:30:00')).mean(dim='time').plot()
```
:::
::::